In [1]:
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#include <math.h>

## SHA-256 Basic Implementation

In [2]:
/*
ROTRIGHT(0b1001, 1) => 0b1100 (bits shifted right, low bits wrap to left)
*/
#define ROTRIGHT(a,b) (((a) >> (b)) | ((a) << (32-(b))))

/*
CH(x,y,z): choose bits from y or z based on x.
For each bit position: if x's bit is 1, take y's bit; otherwise take z's bit.

 MAJ(x,y,z): majority function.
For each bit position: the result bit is the majority value among x,y,z bits.

P0, EP1: the "big sigma" functions, using rotations.
SIG0, SIG1: the "small sigma" functions used in message schedule expansion.
*/
#define CH(x,y,z) (((x) & (y)) ^ (~(x) & (z)))
#define MAJ(x,y,z) (((x) & (y)) ^ ((x) & (z)) ^ ((y) & (z)))
#define EP0(x) (ROTRIGHT(x,2) ^ ROTRIGHT(x,13) ^ ROTRIGHT(x,22))
#define EP1(x) (ROTRIGHT(x,6) ^ ROTRIGHT(x,11) ^ ROTRIGHT(x,25))
#define SIG0(x) (ROTRIGHT(x,7) ^ ROTRIGHT(x,18) ^ ((x) >> 3))
#define SIG1(x) (ROTRIGHT(x,17) ^ ROTRIGHT(x,19) ^ ((x) >> 10))

/*
   ROUND CONSTANTS (k)
   These 64 constants are part of the SHA-256 specification.
   They are the first 32 bits of the fractional part of the cube roots of the first 64 primes.
   They are fixed and chosen to provide diffusion/mixing properties.
*/
static const uint32_t k[64] = {
  0x428a2f98,0x71374491,0xb5c0fbcf,0xe9b5dba5,0x3956c25b,0x59f111f1,0x923f82a4,0xab1c5ed5,
  0xd807aa98,0x12835b01,0x243185be,0x550c7dc3,0x72be5d74,0x80deb1fe,0x9bdc06a7,0xc19bf174,
  0xe49b69c1,0xefbe4786,0x0fc19dc6,0x240ca1cc,0x2de92c6f,0x4a7484aa,0x5cb0a9dc,0x76f988da,
  0x983e5152,0xa831c66d,0xb00327c8,0xbf597fc7,0xc6e00bf3,0xd5a79147,0x06ca6351,0x14292967,
  0x27b70a85,0x2e1b2138,0x4d2c6dfc,0x53380d13,0x650a7354,0x766a0abb,0x81c2c92e,0x92722c85,
  0xa2bfe8a1,0xa81a664b,0xc24b8b70,0xc76c51a3,0xd192e819,0xd6990624,0xf40e3585,0x106aa070,
  0x19a4c116,0x1e376c08,0x2748774c,0x34b0bcb5,0x391c0cb3,0x4ed8aa4a,0x5b9cca4f,0x682e6ff3,
  0x748f82ee,0x78a5636f,0x84c87814,0x8cc70208,0x90befffa,0xa4506ceb,0xbef9a3f7,0xc67178f2
};

typedef struct {
    uint8_t data[64];
    uint32_t datalen;
    unsigned long long bitlen;
    uint32_t state[8];
} SHA256_CTX;


In [3]:
void sha256_transform(SHA256_CTX *ctx, const uint8_t data[]) {
    uint32_t a,b,c,d,e,f,g,h,i,j,t1,t2,m[64];
    for (i=0,j=0; i < 16; ++i, j += 4)
        m[i] = (data[j] << 24) | (data[j+1] << 16) | (data[j+2] << 8) | (data[j+3]);
    for ( ; i < 64; ++i)
        m[i] = SIG1(m[i-2]) + m[i-7] + SIG0(m[i-15]) + m[i-16];
    a = ctx->state[0]; b = ctx->state[1]; c = ctx->state[2]; d = ctx->state[3];
    e = ctx->state[4]; f = ctx->state[5]; g = ctx->state[6]; h = ctx->state[7];
    for (i=0; i<64; ++i) {
        t1 = h + EP1(e) + CH(e,f,g) + k[i] + m[i];
        t2 = EP0(a) + MAJ(a,b,c);
        h = g; g = f; f = e; e = d + t1;
        d = c; c = b; b = a; a = t1 + t2;
    }
    ctx->state[0] += a; ctx->state[1] += b; ctx->state[2] += c; ctx->state[3] += d;
    ctx->state[4] += e; ctx->state[5] += f; ctx->state[6] += g; ctx->state[7] += h;
}

In [4]:
void sha256_init(SHA256_CTX *ctx) {
    ctx->datalen = 0; ctx->bitlen = 0;
    ctx->state[0]=0x6a09e667; ctx->state[1]=0xbb67ae85;
    ctx->state[2]=0x3c6ef372; ctx->state[3]=0xa54ff53a;
    ctx->state[4]=0x510e527f; ctx->state[5]=0x9b05688c;
    ctx->state[6]=0x1f83d9ab; ctx->state[7]=0x5be0cd19;
}

In [5]:
void sha256_update(SHA256_CTX *ctx, const uint8_t data[], size_t len) {
    for (size_t i=0;i<len;++i) {
        ctx->data[ctx->datalen] = data[i];
        ctx->datalen++;
        if (ctx->datalen==64) {
            sha256_transform(ctx,ctx->data);
            ctx->bitlen += 512;
            ctx->datalen = 0;
        }
    }
}

In [6]:
void sha256_final(SHA256_CTX *ctx, uint8_t hash[]) {
    uint32_t i = ctx->datalen;
    if (ctx->datalen<56) {
        ctx->data[i++] = 0x80;
        while (i<56) ctx->data[i++] = 0x00;
    } else {
        ctx->data[i++] = 0x80;
        while (i<64) ctx->data[i++] = 0x00;
        sha256_transform(ctx,ctx->data);
        memset(ctx->data,0,56);
    }
    ctx->bitlen += ctx->datalen*8;
    ctx->data[63]=ctx->bitlen; ctx->data[62]=ctx->bitlen>>8;
    ctx->data[61]=ctx->bitlen>>16; ctx->data[60]=ctx->bitlen>>24;
    ctx->data[59]=ctx->bitlen>>32; ctx->data[58]=ctx->bitlen>>40;
    ctx->data[57]=ctx->bitlen>>48; ctx->data[56]=ctx->bitlen>>56;
    sha256_transform(ctx,ctx->data);
    for (i=0;i<4;++i) {
        hash[i]      = (ctx->state[0] >> (24-i*8)) & 0x000000ff;
        hash[i+4]    = (ctx->state[1] >> (24-i*8)) & 0x000000ff;
        hash[i+8]    = (ctx->state[2] >> (24-i*8)) & 0x000000ff;
        hash[i+12]   = (ctx->state[3] >> (24-i*8)) & 0x000000ff;
        hash[i+16]   = (ctx->state[4] >> (24-i*8)) & 0x000000ff;
        hash[i+20]   = (ctx->state[5] >> (24-i*8)) & 0x000000ff;
        hash[i+24]   = (ctx->state[6] >> (24-i*8)) & 0x000000ff;
        hash[i+28]   = (ctx->state[7] >> (24-i*8)) & 0x000000ff;
    }
}

AEVMEPYHLYCIWAXAFZONHXDKRQPSVBNIDJOKOILLBIFPSXDWVDRQMIHOVENDKCLCQSSPC

I-II-III
ᗺ
1-1-1

## HashTable and Hashing

Hashing is a technique to map **D**ata (keys) into fixed-size tables using a function `#(key)`.

### Example: 
Key = "Alice"

$h(s) = \left( \sum_{i=1}^{n} \operatorname{ASCII}(s_i) \right) \bmod m $

Output = index in the table

In [7]:
unsigned int simple_hash(const char *str, int table_size) {
    unsigned int hash = 0;
    for (int i = 0; str[i] != '\0'; i++) {
        hash += (unsigned int)str[i];
    }
    return hash % table_size;
}

In [8]:
const char *keys[] = {"Alice", "Bob", "Charlie", "David"};
    int size = 10;

    for (int i = 0; i < 4; i++) {
        printf("Key: %s -> Hash Index: %u\n", keys[i], simple_hash(keys[i], size));
    }

Key: Alice -> Hash Index: 8
Key: Bob -> Hash Index: 5
Key: Charlie -> Hash Index: 6
Key: David -> Hash Index: 8


### Collision Handling: Chaining with Linked Lists

In [9]:
#define TABLE_SIZE 7

typedef struct Node {
    char key[50];
    struct Node* next;
} Node;

Node* hashTable[TABLE_SIZE];

In [10]:
void insert(const char *key) {
    unsigned int index = simple_hash(key, TABLE_SIZE);
    Node* newNode = (Node*)malloc(sizeof(Node));
    strcpy(newNode->key, key);
    newNode->next = hashTable[index];
    hashTable[index] = newNode;
}

In [11]:
Node* search(const char *key) {
    unsigned int index = simple_hash(key, TABLE_SIZE);
    Node* curr = hashTable[index];
    while (curr) {
        if (strcmp(curr->key, key) == 0) return curr;
        curr = curr->next;
    }
    return NULL;
}


In [12]:
void printTable() {
    for (int i = 0; i < TABLE_SIZE; i++) {
        printf("[%d]: ", i);
        Node* curr = hashTable[i];
        while (curr) {
            printf("%s -> ", curr->key);
            curr = curr->next;
        }
        printf("NULL\n");
    }
}

In [13]:
insert("Alice");
insert("Bob");
insert("Charlie");
insert("David");
insert("Eve");

printTable();

printf("\nSearching for 'Charlie': %s\n", search("Charlie") ? "Found" : "Not Found");
printf("Searching for 'Oscar': %s\n", search("Oscar") ? "Found" : "Not Found");

[0]: NULL
[1]: Eve -> NULL
[2]: Bob -> Alice -> NULL
[3]: Charlie -> NULL
[4]: NULL
[5]: David -> NULL
[6]: NULL

Searching for 'Charlie': Found
Searching for 'Oscar': Not Found


### Collision Handling: Linear Probing

In [14]:
char* hashTableLP[TABLE_SIZE];

In [15]:
void insertLP(const char *key) {
    unsigned int index = simple_hash(key, TABLE_SIZE);
    int i;
    for (i = 0; i < TABLE_SIZE; i++) {
        int pos = (index + i) % TABLE_SIZE;
        if (hashTableLP[pos] == NULL) {
            hashTableLP[pos] = strdup(key);  // allocate memory for the string
            return;
        }
    }
    printf("Table full! Could not insert %s\n", key);
}

In [16]:
int searchLP(const char *key) {
    unsigned int index = simple_hash(key, TABLE_SIZE);
    int i;
    for (i = 0; i < TABLE_SIZE; i++) {
        int pos = (index + i) % TABLE_SIZE;
        if (hashTableLP[pos] == NULL) return 0;   // empty slot, stop
        if (strcmp(hashTableLP[pos], key) == 0) return 1;
    }
    return 0;
}

In [17]:
void printTableLP() {
    int i;
    for (i = 0; i < TABLE_SIZE; i++) {
        printf("[%d]: %s\n", i, hashTableLP[i] ? hashTableLP[i] : "NULL");
    }
}

In [18]:
int i;
// Initialize table
for (i = 0; i < TABLE_SIZE; i++) hashTableLP[i] = NULL;

// Insert some keys
insertLP("Alice");
insertLP("Bob");
insertLP("Charlie");
insertLP("David");
insertLP("Eve");

// Print table
printTableLP();

// Search
printf("\nSearching for 'Eve': %s\n", searchLP("Eve") ? "Found" : "Not Found");
printf("Searching for 'Oscar': %s\n", searchLP("Oscar") ? "Found" : "Not Found");

[0]: NULL
[1]: Eve
[2]: Alice
[3]: Bob
[4]: Charlie
[5]: David
[6]: NULL

Searching for 'Eve': Found
Searching for 'Oscar': Not Found


## Checksum - Hash

A checksum simply verifies with a high degree of confidence that there wa**s** no corruption causing a copied file to differ from the original. 

In general a checksum provides no guarantee that intentional modifications weren't made, and in many cases it is trivial to change the file while still having the same checksum. 

Examples of checksums are CRCs, Adler-32, XOR (parity byte(s)).

--- 


Cryptographic hashes provide additional properties over simple checksums (all cryptographic hashes can be used as checksums, but not all checksums are cryptographic hashes).

Cryptographic hashes (that aren't broken or weak) provide collision and preimage resistance. Collision resistance means that it isn't feasible to create two files that have the same hash, and preimage resistance means that it isn't feasible to create a file with the same hash as a specific target file.

MD5 and SHA1 are both broken in regard to collisions, but are safe against preimage attacks (due to the birthday paradox collisions are much easier to generate). SHA256 is commonly used today, and is safe against both.

In cryptography, a collision attack on a cryptographic hash tries to find **two inputs producing the same hash value**, i.e. a hash collision. This is in contrast to a preimage attack where **a specific target hash value is known**.

In the context of the data structure, lambda (λ) represents the load factor of a hash table. This is a crucial metric that indicates how full the hash table is. It is calculated as the ratio of the number of entries (N) to the number of buckets or locations (M):

In [19]:
#define N 150    // number of keys
#define M 100   // table size

unsigned int hash_div(int key) {
    return key % M; // division method
}

In [20]:
int table[M];  // counts of keys per bucket

srand(time(NULL));

// Insert N random keys
for (int i = 0; i < N; i++) {
    int key = rand();
    unsigned int idx = hash_div(key);
    table[idx]++;
}

printf("Bucket counts:\n");
for (int i = 0; i < M; i++) printf("Bucket %d -> %d keys\n", i, table[i]);

int freq[20] = {0};

for (int i = 0; i < M; i++) if (table[i] < 20) freq[table[i]]++;


printf("\nDistribution of bucket sizes:\n");
for (int k = 0; k < 20; k++) {
    if (freq[k] > 0) printf("Buckets with %d keys: %d\n", k, freq[k]);
}

Bucket counts:
Bucket 0 -> 2 keys
Bucket 1 -> 0 keys
Bucket 2 -> 0 keys
Bucket 3 -> 2 keys
Bucket 4 -> 1 keys
Bucket 5 -> 2 keys
Bucket 6 -> 2 keys
Bucket 7 -> 0 keys
Bucket 8 -> 1 keys
Bucket 9 -> 2 keys
Bucket 10 -> 2 keys
Bucket 11 -> 0 keys
Bucket 12 -> 3 keys
Bucket 13 -> 1 keys
Bucket 14 -> 3 keys
Bucket 15 -> 0 keys
Bucket 16 -> 2 keys
Bucket 17 -> 3 keys
Bucket 18 -> 1 keys
Bucket 19 -> 1 keys
Bucket 20 -> 0 keys
Bucket 21 -> 1 keys
Bucket 22 -> 2 keys
Bucket 23 -> 0 keys
Bucket 24 -> 3 keys
Bucket 25 -> 0 keys
Bucket 26 -> 0 keys
Bucket 27 -> 1 keys
Bucket 28 -> 2 keys
Bucket 29 -> 1 keys
Bucket 30 -> 1 keys
Bucket 31 -> 1 keys
Bucket 32 -> 0 keys
Bucket 33 -> 0 keys
Bucket 34 -> 1 keys
Bucket 35 -> 2 keys
Bucket 36 -> 4 keys
Bucket 37 -> 2 keys
Bucket 38 -> 1 keys
Bucket 39 -> 0 keys
Bucket 40 -> 2 keys
Bucket 41 -> 2 keys
Bucket 42 -> 5 keys
Bucket 43 -> 2 keys
Bucket 44 -> 1 keys
Bucket 45 -> 2 keys
Bucket 46 -> 1 keys
Bucket 47 -> 3 keys
Bucket 48 -> 2 keys
Bucket 49 -> 2 

In [21]:
// Theoretical Poisson distribution
double lambda = (double)N / M;
printf("\nTheoretical Poisson(Lambda=%.2f):\n", lambda);
int kmax = (int)(lambda + 6*sqrt(lambda) + 1);
for (int k = 0; k <= kmax && k <= N; k++) {
    double prob = exp(-lambda);
    for (int j = 1; j <= k; j++) prob *= lambda / j;
    int expected = (int)(prob * M + 0.5); // expected number of buckets
    printf("k=%2d : %2d expected | ", k, expected);
    for (int j = 0; j < expected; j++) printf("#");
    printf("\n");
}


Theoretical Poisson(Lambda=1.50):
k= 0 : 22 expected | ######################
k= 1 : 33 expected | #################################
k= 2 : 25 expected | #########################
k= 3 : 13 expected | #############
k= 4 :  5 expected | #####
k= 5 :  1 expected | #
k= 6 :  0 expected | 
k= 7 :  0 expected | 
k= 8 :  0 expected | 
k= 9 :  0 expected | 


### Checksum with SHA-1 (and safer alternatives)

> Short note: **SHA-1 is broken for collision resistance** — do not use it for security-sensitive tasks (digital signatures, **c**ertificate fingerprints, etc.). Prefer **SHA-256**. https://shattered.io/

---

#### Windows — `certutil`

```cmd
certutil -hashfile C:\path\to\file SHA1

certutil -hashfile C:\path\to\file SHA256

### Prime numbers

In [22]:
#define TABLE_SIZE_DIV 10000  // table size for division method
#define NUM_KEYS_DIV 1000   // number of random keys

unsigned int hash_division(int key_div) {
    return key_div % TABLE_SIZE_DIV;
}

In [23]:
printf("\nDivision Method Hashing\n");

srand(time(NULL)); // seed for random generator

int keys_div[NUM_KEYS_DIV];
int bucket_counts[TABLE_SIZE_DIV] = {0}; // track occupancy of each bucket

// Generate random keys
for (int i = 0; i < NUM_KEYS_DIV; i++) {
    keys_div[i] = rand(); 
}

// Print each key and its bucket, update counts
for (int i = 0; i < NUM_KEYS_DIV; i++) {
    unsigned int b = hash_division(keys_div[i]);
    printf("Key %d -> bucket %u\n", keys_div[i], b);
    bucket_counts[b]++; 
}

// Print final bucket sizes
printf("\nFinal Bucket Sizes\n");
for (int b = 0; b < TABLE_SIZE_DIV; b++) {
    printf("Bucket %2d : %d keys\n", b, bucket_counts[b]);
}


Division Method Hashing
Key 1830376982 -> bucket 6982
Key 2025653599 -> bucket 3599
Key 375539736 -> bucket 9736
Key 295874385 -> bucket 4385
Key 1202845429 -> bucket 5429
Key 1060729786 -> bucket 9786
Key 1381656983 -> bucket 6983
Key 1316228785 -> bucket 8785
Key 1286889180 -> bucket 9180
Key 2116212875 -> bucket 2875
Key 771613270 -> bucket 3270
Key 1980455488 -> bucket 5488
Key 286955800 -> bucket 5800
Key 1803452400 -> bucket 2400
Key 334365045 -> bucket 5045
Key 1944180298 -> bucket 298
Key 1421014399 -> bucket 4399
Key 658379696 -> bucket 9696
Key 1996423724 -> bucket 3724
Key 407565956 -> bucket 5956
Key 333142967 -> bucket 2967
Key 1191479179 -> bucket 9179
Key 2112256247 -> bucket 6247
Key 422961690 -> bucket 1690
Key 1604369667 -> bucket 9667
Key 244220236 -> bucket 236
Key 374315469 -> bucket 5469
Key 145290830 -> bucket 830
Key 1297573808 -> bucket 3808
Key 295947331 -> bucket 7331
Key 115043062 -> bucket 3062
Key 980467143 -> bucket 7143
Key 174117282 -> bucket 7282
Key 

In [24]:
#define TABLE_SIZE_UNIV 10000   // table size for universal hashing
#define NUM_KEYS_UNIV 1000   // number of random keys
#define PRIME_UNIV 1000019   // large prime > TABLE_SIZE_UNIV

int a_univ, b_univ;  // universal hash parameters

In [25]:
unsigned int hash_universal(int key_univ) {
    return ((long long)a_univ * key_univ + b_univ) % PRIME_UNIV % TABLE_SIZE_UNIV;
}

In [26]:
printf("\nUniversal Hashing Method\n");

srand(time(NULL));

// Choose random parameters a, b
a_univ = 1 + rand() % (PRIME_UNIV - 1); // 1 <= a < p
b_univ = rand() % PRIME_UNIV;           // 0 <= b < p

printf("Hash parameters: a=%d, b=%d\n", a_univ, b_univ);

int keys_univ[NUM_KEYS_UNIV];
int bucket_counts_univ[TABLE_SIZE_UNIV] = {0};

// Generate 1000 random keys
for (int i = 0; i < NUM_KEYS_UNIV; i++) {
    keys_univ[i] = rand();
}

// Print each key and its bucket, update counts
for (int i = 0; i < NUM_KEYS_UNIV; i++) {
    unsigned int b = hash_universal(keys_univ[i]);
    printf("Key %d -> bucket %u\n", keys_univ[i], b);
    bucket_counts_univ[b]++;
}

// Print final bucket sizes
printf("\nFinal Bucket Sizes (Universal Hashing)\n");
for (int b = 0; b < TABLE_SIZE_UNIV; b++) {
    printf("Bucket %2d : %d keys\n", b, bucket_counts_univ[b]);
}


Universal Hashing Method
Hash parameters: a=344043, b=615124
Key 375539736 -> bucket 9875
Key 295874385 -> bucket 3716
Key 1202845429 -> bucket 3518
Key 1060729786 -> bucket 5166
Key 1381656983 -> bucket 135
Key 1316228785 -> bucket 9655
Key 1286889180 -> bucket 626
Key 2116212875 -> bucket 5282
Key 771613270 -> bucket 6815
Key 1980455488 -> bucket 3970
Key 286955800 -> bucket 2603
Key 1803452400 -> bucket 3990
Key 334365045 -> bucket 467
Key 1944180298 -> bucket 572
Key 1421014399 -> bucket 5689
Key 658379696 -> bucket 234
Key 1996423724 -> bucket 2673
Key 407565956 -> bucket 1782
Key 333142967 -> bucket 7473
Key 1191479179 -> bucket 2444
Key 2112256247 -> bucket 1534
Key 422961690 -> bucket 7158
Key 1604369667 -> bucket 7151
Key 244220236 -> bucket 6618
Key 374315469 -> bucket 1061
Key 145290830 -> bucket 9297
Key 1297573808 -> bucket 9492
Key 295947331 -> bucket 7570
Key 115043062 -> bucket 3638
Key 980467143 -> bucket 1761
Key 174117282 -> bucket 5064
Key 490582798 -> bucket 8389


In [27]:
void analyze_distribution_eval(const char *title, int bucket_counts[], int table_size, int num_keys) {
    double mean = (double)num_keys / table_size;
    int max_load = 0;
    double variance = 0.0;
    double chi_square = 0.0;

    for (int i = 0; i < table_size; i++) {
        if (bucket_counts[i] > max_load) {
            max_load = bucket_counts[i];
        }
        double diff = bucket_counts[i] - mean;
        variance += diff * diff;
        chi_square += (diff * diff) / mean;
    }
    variance /= table_size;

    printf("\n%s \n", title);
    printf("Average load (expected): %.2f\n", mean);
    printf("Maximum load (worst-case collisions): %d\n", max_load);
    printf("Variance of distribution: %.2f\n", variance);
    printf("Chi-square statistic: %.2f\n", chi_square);
}

In [28]:
analyze_distribution_eval("Division Method", bucket_counts, TABLE_SIZE_DIV, NUM_KEYS_DIV);
analyze_distribution_eval("Universal Hashing", bucket_counts_univ, TABLE_SIZE_UNIV, NUM_KEYS_UNIV);


Division Method 
Average load (expected): 0.10
Maximum load (worst-case collisions): 2
Variance of distribution: 0.10
Chi-square statistic: 10040.00

Universal Hashing 
Average load (expected): 0.10
Maximum load (worst-case collisions): 3
Variance of distribution: 0.10
Chi-square statistic: 9900.00


### Salting

In [29]:
void print_hex(const unsigned char *hash, size_t len) {
    for (size_t i=0; i<len; i++) printf("%02x", hash[i]);
    printf("\n");
}

In [30]:
void hash_with_salt(const char *password, const char *salt, unsigned char out_hash[32]) {
    SHA256_CTX ctx;
    sha256_init(&ctx);
    sha256_update(&ctx, (const unsigned char*)password, strlen(password));
    sha256_update(&ctx, (const unsigned char*)salt, strlen(salt));
    sha256_final(&ctx, out_hash);
}

In [31]:
const char *magic_salt_token = "XyZ123!Salt";
const char *signup_spellword = "WrongPass";

unsigned char stored_hash[32];
hash_with_salt(signup_spellword, magic_salt_token, stored_hash);

printf("Stored salted SHA-256 hash: ");
print_hex(stored_hash, 32);

const char *login_try1 = "WrongPass";
const char *login_try2 = "Secret123";
unsigned char attempt[32];

hash_with_salt(login_try1, magic_salt_token, attempt);
printf("Login attempt 1: %s\n", memcmp(stored_hash, attempt, 32)==0 ? "valid" : "invalid");

hash_with_salt(login_try2, magic_salt_token, attempt);
printf("Login attempt 2: %s\n", memcmp(stored_hash, attempt, 32)==0 ? "valid" : "invalid");

Stored salted SHA-256 hash: 8dc158498920871098b50255f985e1e0d3b9c083ff38bbf4581f242eb6082f98
Login attempt 1: valid
Login attempt 2: invalid


### Other Hashing Methods

#### Cuckoo Hashing

In [32]:
#define TABLE_SIZE_C 11  // Hash table size
#define MAX_RECURSION_DEPTH 10  // Maximum allowed number of kicks during insertion

In [33]:
// Two hash tables
typedef struct {
    int* table1;  
    int* table2;
} CuckooHashTable;

In [34]:
#define HASH1(key) ((key) % TABLE_SIZE_C)
#define HASH2(key) (((key) / TABLE_SIZE) % TABLE_SIZE_C)

In [35]:
CuckooHashTable* create_table() {
    CuckooHashTable* hashTable = (CuckooHashTable*) malloc(sizeof(CuckooHashTable));
    hashTable->table1 = (int*) malloc(TABLE_SIZE_C * sizeof(int));
    hashTable->table2 = (int*) malloc(TABLE_SIZE_C * sizeof(int));

    // Initialize tables to a sentinel value (-1 is empty slot)
    for (int i = 0; i < TABLE_SIZE_C; i++) {
        hashTable->table1[i] = -1;
        hashTable->table2[i] = -1;
    }
    return hashTable;
}


In [36]:
// Recursive insertion logic
int insert(CuckooHashTable* hashTable, int key, int depth, int inTable) {
    if (depth > MAX_RECURSION_DEPTH) {
        printf("Insertion failed: Too many recursions\n");
        return 0;  // Max recursion depth
    }

    // Inserting into table1 if its table1 insertion
    if (inTable == 1) {
        int pos1 = HASH1(key);
        if (hashTable->table1[pos1] == -1) {
            hashTable->table1[pos1] = key;  // Insert key if the spot is empty
            return 1;  // Successfully inserted
        } else {
            // If there's a collision, "kick out" existing key
            int oldKey = hashTable->table1[pos1];
            hashTable->table1[pos1] = key;
            printf("Kicked out %d and inserting %d in table1\n", oldKey, key);
            // Recursively insert the kicked-out key into table2
            return insert(hashTable, oldKey, depth + 1, 2);
        }
    } 
    // Try inserting into table2 if its table2 insertion
    else {
        int pos2 = HASH2(key);
        if (hashTable->table2[pos2] == -1) {
            hashTable->table2[pos2] = key;  // Insert key if the spot is empty
            return 1;  // Successfully inserted
        } else {
            // If there's a collision, "kick out" existing key
            int oldKey = hashTable->table2[pos2];
            hashTable->table2[pos2] = key;
            printf("Kicked out %d and inserting %d in table2\n", oldKey, key);
            // Recursively insert the kicked-out key into table1
            return insert(hashTable, oldKey, depth + 1, 1);
        }
    }
}

In [37]:
int cuckoo_insert(CuckooHashTable* hashTable, int key) {
    return insert(hashTable, key, 0, 1);  // Start with table1
}

In [38]:
int search_in_table1(CuckooHashTable* hashTable, int key) {
    int pos1 = HASH1(key);
    if (hashTable->table1[pos1] == key) {
        return 1;  // Found key in table1
    }
    return 0;
}

In [39]:
int search_in_table2(CuckooHashTable* hashTable, int key) {
    int pos2 = HASH2(key);
    if (hashTable->table2[pos2] == key) {
        return 1;  // Found key in table2
    }
    return 0;
}

In [40]:
int cuckoo_search(CuckooHashTable* hashTable, int key) {
    if (search_in_table1(hashTable, key) || search_in_table2(hashTable, key)) return 1; 
    return 0;
}

In [41]:
void display_table(CuckooHashTable* hashTable) {
    printf("Table 1: ");
    for (int i = 0; i < TABLE_SIZE_C; i++) printf("%d ", hashTable->table1[i]);

    printf("\n");
    
    printf("Table 2: ");
    for (int i = 0; i < TABLE_SIZE_C; i++) printf("%d ", hashTable->table2[i]);
    printf("\n");
}

In [42]:
CuckooHashTable* cuckooHashTable = create_table();  

cuckoo_insert(cuckooHashTable, 10);
cuckoo_insert(cuckooHashTable, 22);
cuckoo_insert(cuckooHashTable, 35);
cuckoo_insert(cuckooHashTable, 48);
cuckoo_insert(cuckooHashTable, 59);
cuckoo_insert(cuckooHashTable, 23);
cuckoo_insert(cuckooHashTable, 33);
cuckoo_insert(cuckooHashTable, 43);


if (cuckoo_search(cuckooHashTable, 22)) {
    printf("Key 22 found in the table.\n");
} else printf("Key 22 not found in the table.\n");

if (cuckoo_search(cuckooHashTable, 38)) {
    printf("Key 38 found in the table.\n");
} else printf("Key 38 not found in the table.\n");


display_table(cuckooHashTable);

Kicked out 48 and inserting 59 in table1
Kicked out 22 and inserting 33 in table1
Kicked out 10 and inserting 43 in table1
Key 22 found in the table.
Key 38 not found in the table.
Table 1: 33 23 35 -1 59 -1 -1 -1 -1 -1 43 
Table 2: -1 10 -1 22 -1 -1 48 -1 -1 -1 -1 
